In [1]:
import time
from collections import deque

class HitCounter:
    """Optimized Sliding Window Counter using time buckets."""
    def __init__(self, window_size_seconds=300):
        self.window_size = window_size_seconds
        # Deque stores lists in the format: [timestamp_sec, count]
        self.buckets = deque()
        self.total_count = 0

    def record(self):
        """Record a query in the current second's bucket."""
        now = int(time.time())
        self._evict_old(now)

        # If the last bucket is the current second, increment it.
        # Otherwise, append a new bucket for this second.
        if self.buckets and self.buckets[-1][0] == now:
            self.buckets[-1][1] += 1
        else:
            self.buckets.append([now, 1])

        self.total_count += 1

    def get_rate(self):
        """Calculate the average QPS."""
        now = int(time.time())
        self._evict_old(now)

        if not self.buckets:
            return 0.0

        time_span = now - self.buckets[0][0]
        # Avoid division by zero and cap the span to the actual window size
        effective_span = max(1, min(time_span, self.window_size))

        return self.total_count / effective_span

    def _evict_old(self, current_time):
        """Remove buckets that fall outside the time window."""
        cutoff = current_time - self.window_size
        while self.buckets and self.buckets[0][0] <= cutoff:
            removed = self.buckets.popleft()
            self.total_count -= removed[1]

class KVStore:
    def __init__(self, window_size_seconds=300):
        self.store = {}
        self.window_size = window_size_seconds

        # 1. Created separate counters for GET and PUT
        self.put_counter = HitCounter(window_size_seconds)
        self.get_counter = HitCounter(window_size_seconds)

    def put(self, key, value):
        """Store a key-value pair and record a put query."""
        self.store[key] = value
        self.put_counter.record()

    def get(self, key):
        """Get value by key and record a get query."""
        self.get_counter.record()
        return self.store.get(key)

    def delete(self, key):
        """Delete a key."""
        if key in self.store:
            del self.store[key]
        # Optional: You can create a self.delete_counter if you want to track this too.

    def get_avg_load(self):
        """Return the average QPS for get and put operations."""
        return {
            "avg_put_qps": self.put_counter.get_rate(),
            "avg_get_qps": self.get_counter.get_rate()
        }

    def set_window_size(self, seconds):
        """Adjust the QPS calculation window for all counters."""
        self.window_size = seconds
        self.put_counter.window_size = seconds
        self.get_counter.window_size = seconds

In [4]:
#############OPTIMIZED*****

import time

class HitCounter:
    """
    A circular buffer to count events over a rolling window.
    (Not thread-safe)
    """
    def __init__(self, window_size_seconds=600): # Default 10 minutes
        self.window_size = window_size_seconds

        # Buckets store the count of hits for each second index
        self.buckets = [0] * window_size_seconds

        # Tracks the exact timestamp (in seconds) stored in each bucket
        self.last_reset_time = [0] * window_size_seconds

    def hit(self):
        """Record a single event at the current time."""
        current_time = int(time.time())
        idx = current_time % self.window_size

        # Lazy Reset: If this bucket holds data from an old cycle, wipe it.
        if self.last_reset_time[idx] != current_time:
            self.buckets[idx] = 0
            self.last_reset_time[idx] = current_time

        self.buckets[idx] += 1

    def get_rate(self):
        """Calculate the average rate (hits per second) over the window."""
        current_time = int(time.time())
        total_hits = 0

        for i in range(self.window_size):
            # Only include buckets that fall within the valid window [now - window, now]
            if current_time - self.last_reset_time[i] < self.window_size:
                total_hits += self.buckets[i]

        return total_hits / self.window_size



class KVStore:
    def __init__(self, window_size=600):
        self.store = {}
        # Initialize separate monitors for PUT and GET
        self.put_counter = HitCounter(window_size_seconds=window_size)
        self.get_counter = HitCounter(window_size_seconds=window_size)


    def put(self, key, value):
        self.store[key] = value
        self.put_counter.hit()

    def get(self, key):
        val = self.store.get(key)
        self.get_counter.hit()
        return val

    def get_avg_load(self):
        return {
            "avg_put_qps": self.put_counter.get_rate(),
            "avg_get_qps": self.get_counter.get_rate()
        }

# --- Example Usage ---
if __name__ == "__main__":
    kv = KVStore(window_size=10)

    # Simulate traffic
    for _ in range(50):
        kv.put("user:1", "data")
        kv.get("user:1")
        time.sleep(0.01)

    print(f"Current Load: {kv.get_avg_load()}")

Current Load: {'avg_put_qps': 5.0, 'avg_get_qps': 5.0}
